# **Feature engineering**

[Загрузка данных](#Loading-data)

- [age](#age)
- [MonthlyIncome](#MonthlyIncome)
- [RevolvingUtilizationOfUnsecuredLines](#RevolvingUtilizationOfUnsecuredLines)
- [DebtRatio](#DebtRatio)
- [NumberOfDependents](#NumberOfDependents)
- [NumberOfOpenCreditLinesAndLoans](#NumberOfOpenCreditLinesAndLoans)
- [NumberRealEstateLoansOrLines](#NumberRealEstateLoansOrLines)
- [NumberOfTime30-59DaysPastDueNotWorse](#NumberOfTime30-59DaysPastDueNotWorse)
- [NumberOfTime60-89DaysPastDueNotWorse](#NumberOfTime60-89DaysPastDueNotWorse)
- [NumberOfTimes90DaysLate](#NumberOfTimes90DaysLate)

[Результирующие признаки](#Resulting-DataFrames)

---

## Loading data

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

import random
import pandas  as pd
import numpy   as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from optbinning import OptimalBinning

from sklearn.pipeline      import Pipeline
from sklearn.compose       import ColumnTransformer
from sklearn.preprocessing import scale, robust_scale, power_transform

from src import *

set_seeds()
pd.set_option('display.max_columns', None)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
train_raw = pd.read_csv('../data/raw/cs-training.csv').drop(columns=['Unnamed: 0'])
print(f"Data shape: {train_raw.shape}")

X = train_raw.drop(columns=['SeriousDlqin2yrs']).copy()
y = train_raw['SeriousDlqin2yrs'].copy()
print(f"Data shape: {train_raw.shape}")

X = train_raw.drop(columns=['SeriousDlqin2yrs']).copy()
y = train_raw['SeriousDlqin2yrs'].copy()

In [ ]:
# DataFrames for WoE and non-WoE preprocessed features

train_woe       = train_raw.copy()
train_processed = train_raw.copy()

---

## `age`

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'age',
                      n_quantiles=40, discrete=True, title='', ax=axs[0])
plot_WoE(train_raw, 'age', ax=axs[1])

fig.suptitle(fr'raw')
plt.tight_layout()

In [ ]:
AGE_CLIP_UPPER_PCTL = 0.99

AGE_CLIP_UPPER = train_raw.age.quantile(AGE_CLIP_UPPER_PCTL)
AGE_CLIP_LOWER = 21

print(f'{AGE_CLIP_UPPER_PCTL * 100} процентиль: {AGE_CLIP_UPPER}')

### Preprocessing

In [ ]:
train_processed['age'] = scale(train_raw['age'].clip(lower=AGE_CLIP_LOWER, upper=AGE_CLIP_UPPER))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'age',
                      title=rf'raw', discrete=True, ax=axes[0])
plot_feature_analysis(train_processed, 'age', y, bins=train_processed['age'].unique().size,
                      title=rf'preprocessed', ax=axes[1])

plt.tight_layout()

### New Features

- Учитывая немонотонную зависимость риска дефолта от возраста заёмщиков, целесообразно добавить полиномиальные признаки `age^2` и `age^3`, чтобы линейные модели могли лучше аппроксимировать нелинейный паттерн.
- Для моделей на основе деревьев могут быть поезны бинарные признаки, указывающие на принадлежность к молодым и пожилым группам заёмщиков.

In [ ]:
YOUNG_THRESHOLD  = 30
SENIOR_THRESHOLD = 59

train_processed['is_young']  = train_raw['age'].lt(YOUNG_THRESHOLD).astype(int)
train_processed['is_senior'] = train_raw['age'].gt(SENIOR_THRESHOLD).astype(int)

train_processed['age^2'] = scale(train_raw['age'].clip(lower=AGE_CLIP_LOWER, upper=AGE_CLIP_UPPER) ** 2)
train_processed['age^3'] = scale(train_raw['age'].clip(lower=AGE_CLIP_LOWER, upper=AGE_CLIP_UPPER) ** 3)

In [ ]:
age_new_features = ['is_young', 'is_senior', 'age^2', 'age^3']
train_processed[age_new_features].describe(percentiles=[])

### WoE Transformation

In [ ]:
train_woe['age'], age_binner = apply_woe_binning(
    df=train_raw,
    feature='age',
    target=train_raw['SeriousDlqin2yrs'],
    max_n_bins=10,
    monotonic_trend='descending'
)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

plot_WoE(train_raw, 'age', 'SeriousDlqin2yrs', ax=axs[0], title='raw',
         xlabel='age');

plot_WoE(train_woe, 'age', 'SeriousDlqin2yrs', ax=axs[1],
         title='preprocessed', xlabel='age');

In [ ]:
binning_table = age_binner.binning_table
binning_table.build()

In [ ]:
binning_table.plot(figsize=(6, 4))

## `MonthlyIncome`

In [ ]:
INCOME_CLIP_UPPER_PCTL      = 0.90
INCOME_THRESHOLD_LOWER_PCTL = 0.10

INCOME_CLIP_UPPER      = train_raw['MonthlyIncome'].quantile(INCOME_CLIP_UPPER_PCTL)
INCOME_THRESHOLD_LOWER = train_raw['MonthlyIncome'].quantile(INCOME_THRESHOLD_LOWER_PCTL)

print(f'Low  income threshold: {INCOME_THRESHOLD_LOWER:.2f}')
print(f'High income threshold: {INCOME_CLIP_UPPER:.2f}')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'MonthlyIncome', max_value=INCOME_CLIP_UPPER*2,
                      n_quantiles=40, title='', ax=axs[0])
plot_WoE(train_raw.loc[train_raw.MonthlyIncome.le(INCOME_CLIP_UPPER*2)], 'MonthlyIncome', ax=axs[1])

fig.suptitle(fr'raw $\leq ({INCOME_CLIP_UPPER_PCTL}$ percentile) * 2')
plt.tight_layout()

### Preprocessing

In [ ]:
train_processed['MonthlyIncome'] = scale(train_raw['MonthlyIncome'].fillna(0.0).clip(upper=INCOME_CLIP_UPPER)) # .where(train_raw['MonthlyIncome'].le(INCOME_CLIP_UPPER), 0.0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'MonthlyIncome', max_value=INCOME_CLIP_UPPER*2,
                      title=rf'raw $\leq ({INCOME_CLIP_UPPER_PCTL}$ percentile) * 2', ax=axes[0])
plot_feature_analysis(train_processed, 'MonthlyIncome', y,
                      title=rf'preprocessed', ax=axes[1])

plt.tight_layout()

### New Features

- Учитывая немонотонную зависимость риска дефолта от дохода заёмщиков, целесообразно добавить полиномиальные признаки `MonthlyIncome^2` и `MonthlyIncome^3`, чтобы линейные модели могли лучше аппроксимировать нелинейный паттерн.
- Добавим также и бинарные индикаторы:
  - Индикатор отсутствующих значений,
  - Индикатор очень высоких доходов (> 90 процентиля),
  - Индикатор нулевых доходов,
  - Индикатор очень низких доходов (< 10 процентиля).

In [ ]:
train_processed['MonthlyIncome_missing'] = train_raw['MonthlyIncome'].isna().astype(int)
train_processed['MonthlyIncome_clipped'] = train_raw['MonthlyIncome'].gt(INCOME_CLIP_UPPER).astype(int)
train_processed['MonthlyIncome_zero']    = train_raw['MonthlyIncome'].eq(0.0).astype(int)
train_processed['MonthlyIncome^2']       = train_raw['MonthlyIncome'] ** 2
train_processed['MonthlyIncome^3']       = train_raw['MonthlyIncome'] ** 3
train_processed['LowIncome']             = train_raw['MonthlyIncome'].le(INCOME_THRESHOLD_LOWER).astype(int)

In [ ]:
income_new_features = ['MonthlyIncome_missing', 'MonthlyIncome_zero', 'MonthlyIncome_clipped', 'LowIncome']
train_processed[income_new_features].describe(percentiles=[])

### WoE Transformation

In [ ]:
train_woe['MonthlyIncome'] = train_raw['MonthlyIncome'].fillna(0.0)

In [ ]:
train_woe['MonthlyIncome'], income_binner = apply_woe_binning(
    df=train_woe,
    feature='MonthlyIncome',
    target=train_raw['SeriousDlqin2yrs'],
    max_n_bins=10,
    monotonic_trend='auto'
)

In [ ]:
binning_table = income_binner.binning_table
binning_table.build()

In [ ]:
binning_table.plot(figsize=(6, 4))

## `RevolvingUtilizationOfUnsecuredLines`

In [ ]:
UTIL_CLIP_UPPER_PCTL = 0.99

UTIL_CLIP_UPPER = train_raw['RevolvingUtilizationOfUnsecuredLines'].quantile(UTIL_CLIP_UPPER_PCTL)
print(f'{100*UTIL_CLIP_UPPER_PCTL} процентиль: {UTIL_CLIP_UPPER}')

UTIL_CLIP_UPPER = 1.0

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'RevolvingUtilizationOfUnsecuredLines',
                      n_quantiles=100, max_value=UTIL_CLIP_UPPER*2, title='', ax=axs[0])
plot_WoE(train_raw, 'RevolvingUtilizationOfUnsecuredLines', clip_max=UTIL_CLIP_UPPER*2, ax=axs[1])

fig.suptitle(fr'raw $\leq {UTIL_CLIP_UPPER*2}$')
plt.tight_layout()

### Preprocessing

In [ ]:
train_processed['RevolvingUtilizationOfUnsecuredLines'] = \
    scale(train_raw['RevolvingUtilizationOfUnsecuredLines'].clip(upper=UTIL_CLIP_UPPER))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_processed, 'RevolvingUtilizationOfUnsecuredLines',
                      n_quantiles=100, title='', ax=axs[0])
plot_WoE(train_processed, 'RevolvingUtilizationOfUnsecuredLines', ax=axs[1])

fig.suptitle('preprocessed')
plt.tight_layout()

### New Features

- Добавим бинарные индикаторы, чтобы подчеркнуть принадлежность клиентов к группам:
  - Индикатор очень высоких значений (> 99 процентиля);
  - Индикатор значений = 0.99999990, коих порядка 7%.

In [ ]:
train_processed['utilization_clipped'] = \
    train_raw['RevolvingUtilizationOfUnsecuredLines'].gt(UTIL_CLIP_UPPER).astype(int)

train_processed['utilization_99999990'] = \
    train_raw['RevolvingUtilizationOfUnsecuredLines'].eq(0.9999999).astype(int)

In [ ]:
util_new_features = ['utilization_clipped', 'utilization_99999990']
train_processed[util_new_features].describe(percentiles=[])

### WoE Transformation

In [ ]:
train_woe['RevolvingUtilizationOfUnsecuredLines'], util_binner = apply_woe_binning(
    df=train_raw,
    feature='RevolvingUtilizationOfUnsecuredLines',
    target=train_raw['SeriousDlqin2yrs'],
    max_n_bins=10,
    monotonic_trend='ascending'
)

In [ ]:
binning_table = util_binner.binning_table
binning_table.build()

In [ ]:
binning_table.plot(figsize=(6, 4))

## `DebtRatio`

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'DebtRatio',
                      max_value=2, title='', ax=axs[0])
plot_WoE(train_raw.loc[train_raw.DebtRatio.le(2)], 'DebtRatio', ax=axs[1])

fig.suptitle(fr'raw $\leq 2$')
plt.tight_layout()
plt.show()

In [ ]:
lower = 2
upper = 20

fig, ax1 = plt.subplots(figsize=(8, 4))

sns.histplot(
    x=train_raw.loc[train_raw.DebtRatio.between(left=lower, right=upper), 'DebtRatio'],
    element='step', 
    ax=ax1, 
    bins=200
)

test_whole = train_raw.loc[
    (train_raw.DebtRatio % 1 == 0) & 
    (train_raw.DebtRatio.between(lower, upper)),
    ['DebtRatio', 'SeriousDlqin2yrs']
].astype(int)

ax2 = ax1.twinx()
plot_WoE(
    test_whole, 
    'DebtRatio', 
    'SeriousDlqin2yrs', 
    edgecolors=None,
    ax=ax2, 
    bins=18, 
    color='orange', 
    title='WoE на целых значениях'
)

ax2.grid(True, alpha=0.4)
ax1.set_xticks(np.arange(lower, upper + 1, 2));

На целых значениях риск тоже немонотонен

### Preprocessing

Из-за природы фичи, имеет смысл попробовать её и без предобработки

In [ ]:
DEBT_CLIP_UPPER = 4.0

In [ ]:
custom_median = train_raw.loc[train_raw.DebtRatio.le(DEBT_CLIP_UPPER), 'DebtRatio'].median()
DebtRatio_clipped = train_raw[['DebtRatio']].where(train_raw.DebtRatio.le(DEBT_CLIP_UPPER),
                                              custom_median).clip(upper=DEBT_CLIP_UPPER)

train_processed['DebtRatio'] = power_transform(DebtRatio_clipped, method='yeo-johnson')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'DebtRatio', max_value=2,
                      title=rf'raw $\leq {2}$', ax=axes[0])
plot_feature_analysis(train_processed, 'DebtRatio',
                      title=rf'preprocessed', ax=axes[1])

plt.tight_layout()

### New Features

- Учитывая немонотонную зависимость риска дефолта от долговой нагрузки, целесообразно добавить полиномиальные признаки `DebtRatio^2` и `DebtRatio^3`, чтобы линейные модели могли лучше аппроксимировать нелинейный паттерн.
- Также добавим бинарные индикаторы:
  - Индикатор очень низких значений (меньше медианы значений, которые < 4).
  - Индикатор очень высоких значений (> 4),
  - Индикатор нулевых значений,
  - Индикатор целых значений,

In [ ]:
train_processed['DebtRatio^2'] = power_transform(DebtRatio_clipped ** 2, method='yeo-johnson')
train_processed['DebtRatio^3'] = power_transform(DebtRatio_clipped ** 3, method='yeo-johnson')

train_processed['DebtRatio_low'] = train_raw['DebtRatio'].lt(custom_median).astype(int)
train_processed['DebtRatio_clipped'] = train_raw['DebtRatio'].gt(DEBT_CLIP_UPPER).astype(int)

train_processed['DebtRatio_zero']  = train_raw['DebtRatio'].eq(0.0).astype(int)
train_processed['DebtRatio_whole'] = ((np.isclose(train_raw['DebtRatio'] % 1, 0, atol=1e-9)) &
                                      (train_raw['DebtRatio'].gt(0))).astype(int)

### WoE Transformation

In [ ]:
train_woe['DebtRatio'], debt_binner = apply_woe_binning(
    df=train_raw,
    feature='DebtRatio',
    target=train_raw['SeriousDlqin2yrs'],
    max_n_bins=6,
    special_codes={'zero': 0},
)

In [ ]:
binning_table = debt_binner.binning_table
binning_table.build()

In [ ]:
binning_table.plot(figsize=(6, 4))

## `NumberOfDependents`

In [ ]:
DEPS_CLIP_UPPER_PCTL = 0.975
DEPS_CLIP_UPPER = train_raw.NumberOfDependents.quantile(DEPS_CLIP_UPPER_PCTL)

print(f'{100*DEPS_CLIP_UPPER_PCTL} процентиль: {DEPS_CLIP_UPPER}')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'NumberOfDependents', discrete=True,
                      title='', ax=axs[0])
plot_WoE(train_raw, 'NumberOfDependents', ax=axs[1])

fig.suptitle(fr'raw')
plt.tight_layout()

### New Features

Добавим индикатор отсутствующих значений, чтобы не потерять информацию после восполнения пропусков.

In [ ]:
train_processed['NumberOfDependents_missing'] = train_raw['NumberOfDependents'].isna().astype(int)

### Preprocessing

In [ ]:
train_processed['NumberOfDependents'] = scale(train_raw['NumberOfDependents'].fillna(0.0).clip(upper=DEPS_CLIP_UPPER))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'NumberOfDependents', max_value=8,
                      title=rf'raw $\leq 8$', discrete=True, ax=axes[0])
plot_feature_analysis(train_processed, 'NumberOfDependents', discrete=True,
                      title=rf'preprocessed', ax=axes[1])

plt.tight_layout()
plt.show()

### WoE Transformation

In [ ]:
train_woe['NumberOfDependents'], deps_binner = apply_woe_binning(
    df=train_raw,
    feature='NumberOfDependents',
    target=train_raw['SeriousDlqin2yrs'],
    max_n_bins=10,
    min_prebin_size=0.02,
    monotonic_trend='ascending'
)

In [ ]:
binning_table = deps_binner.binning_table
binning_table.build()

In [ ]:
binning_table.plot(figsize=(6, 4))

## `NumberOfOpenCreditLinesAndLoans`

In [ ]:
LOANS_CLIP_UPPER_PCTL = 0.99
LOANS_CLIP_UPPER = train_raw['NumberOfOpenCreditLinesAndLoans'].quantile(LOANS_CLIP_UPPER_PCTL)

print(f'{100*LOANS_CLIP_UPPER_PCTL} процентиль: {LOANS_CLIP_UPPER}')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'NumberOfOpenCreditLinesAndLoans', discrete=True,
                      max_value=LOANS_CLIP_UPPER, title='', ax=axs[0])
plot_WoE(train_raw.loc[train_raw['NumberOfOpenCreditLinesAndLoans'].le(LOANS_CLIP_UPPER)], 'NumberOfOpenCreditLinesAndLoans', ax=axs[1])

fig.suptitle(fr'raw $\leq {100*LOANS_CLIP_UPPER_PCTL}$ percentile')
plt.tight_layout()

### Preprocessing

In [ ]:
loans_clipped = train_raw['NumberOfOpenCreditLinesAndLoans'].clip(upper=LOANS_CLIP_UPPER)

train_processed['NumberOfOpenCreditLinesAndLoans'] = scale(np.log1p(loans_clipped))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'NumberOfOpenCreditLinesAndLoans', max_value=LOANS_CLIP_UPPER,
                      title=fr'raw $\leq {100*LOANS_CLIP_UPPER_PCTL}$ percentile', discrete=True, ax=axes[0])
plot_feature_analysis(train_processed, 'NumberOfOpenCreditLinesAndLoans', discrete=True,
                      title=rf'preprocessed', ax=axes[1])

plt.tight_layout()

### New Features

Для лучшей аппроксимации нелинейного паттерна линейными моделями, добавим полиномиальные признаки:

In [ ]:
train_processed['NumberOfOpenCreditLinesAndLoans^2'] = scale(np.log1p(loans_clipped ** 2))
train_processed['NumberOfOpenCreditLinesAndLoans^3'] = scale(np.log1p(loans_clipped ** 3))

### WoE Transformation

In [ ]:
train_woe['NumberOfOpenCreditLinesAndLoans'], loans_binner = apply_woe_binning(
    df=train_raw,
    feature='NumberOfOpenCreditLinesAndLoans',
    target=train_raw['SeriousDlqin2yrs'],
    max_n_bins=10,
    monotonic_trend='descending'
)

In [ ]:
binning_table = loans_binner.binning_table
binning_table.build()

In [ ]:
binning_table.plot(figsize=(6, 4))

## `NumberRealEstateLoansOrLines`

In [ ]:
ESTATE_CLIP_UPPER_PCTL = 0.99
ESTATE_CLIP_UPPER = train_raw['NumberRealEstateLoansOrLines'].quantile(ESTATE_CLIP_UPPER_PCTL)

print(f'{100*ESTATE_CLIP_UPPER_PCTL} процентиль: {ESTATE_CLIP_UPPER}')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'NumberRealEstateLoansOrLines', discrete=True,
                      title='', ax=axs[0])
plot_WoE(train_raw, 'NumberRealEstateLoansOrLines', ax=axs[1])

fig.suptitle(fr'raw')
plt.tight_layout()

### Preprocessing

In [ ]:
estate_clipped = train_raw['NumberRealEstateLoansOrLines'].clip(upper=ESTATE_CLIP_UPPER)

train_processed['NumberRealEstateLoansOrLines'] = scale(estate_clipped)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'NumberRealEstateLoansOrLines', max_value=ESTATE_CLIP_UPPER,
                      title=rf'raw $\leq {ESTATE_CLIP_UPPER}$', discrete=True, ax=axes[0])
plot_feature_analysis(train_processed, 'NumberRealEstateLoansOrLines', discrete=True,
                      title=rf'preprocessed', ax=axes[1])

plt.tight_layout()

### New Features

Для лучшей аппроксимации нелинейного паттерна линейными моделями, добавим полиномиальные признаки:

In [ ]:
train_processed['NumberRealEstateLoansOrLines^2'] = scale(estate_clipped ** 2)
train_processed['NumberRealEstateLoansOrLines^3'] = scale(estate_clipped ** 3)

### WoE Transformation

In [ ]:
train_woe['NumberRealEstateLoansOrLines'], estate_binner = apply_woe_binning(
    df=train_raw,
    feature='NumberRealEstateLoansOrLines',
    target=train_raw['SeriousDlqin2yrs'],
    max_n_bins=10,
    monotonic_trend='auto'
)

In [ ]:
binning_table = estate_binner.binning_table
binning_table.build()

In [ ]:
binning_table.plot(figsize=(6, 4))

## `NumberOfTime30-59DaysPastDueNotWorse`

In [ ]:
PD30_59_CLIP_UPPER_PCTL = 0.99
PD30_59_CLIP_UPPER = train_raw['NumberOfTime30-59DaysPastDueNotWorse'].quantile(PD30_59_CLIP_UPPER_PCTL)

print(f'{100*PD30_59_CLIP_UPPER_PCTL} процентиль: {PD30_59_CLIP_UPPER}')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'NumberOfTime30-59DaysPastDueNotWorse', discrete=True,
                      title='', ax=axs[0])
plot_WoE(train_raw,
         'NumberOfTime30-59DaysPastDueNotWorse', ax=axs[1])

fig.suptitle(fr'raw')
plt.tight_layout()

### Preprocessing

In [ ]:
train_processed['NumberOfTime30-59DaysPastDueNotWorse'] = \
    scale(train_raw['NumberOfTime30-59DaysPastDueNotWorse'].clip(upper=PD30_59_CLIP_UPPER))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'NumberOfTime30-59DaysPastDueNotWorse', max_value=PD30_59_CLIP_UPPER*2,
                      title=rf'raw $\leq {PD30_59_CLIP_UPPER*2}$', discrete=True, ax=axes[0])
plot_feature_analysis(train_processed, 'NumberOfTime30-59DaysPastDueNotWorse', discrete=True,
                      title=rf'preprocessed', ax=axes[1])

plt.tight_layout()

### WoE Transformation

In [ ]:
train_woe['NumberOfTime30-59DaysPastDueNotWorse'], pd30_59_binner = apply_woe_binning(
    df=train_raw,
    feature='NumberOfTime30-59DaysPastDueNotWorse',
    target=train_raw['SeriousDlqin2yrs'],
    max_n_bins=10,
    min_prebin_size=0.005,
    special_codes={98: 98, 96: 96},
    monotonic_trend='ascending'
)

In [ ]:
binning_table = pd30_59_binner.binning_table
binning_table.build()

In [ ]:
binning_table.plot(figsize=(6, 4))

## `NumberOfTime60-89DaysPastDueNotWorse`

In [ ]:
PD60_89_CLIP_UPPER_PCTL = 0.99
PD60_89_CLIP_UPPER = train_raw['NumberOfTime60-89DaysPastDueNotWorse'].quantile(PD60_89_CLIP_UPPER_PCTL)

print(f'{100*PD60_89_CLIP_UPPER_PCTL} процентиль: {PD60_89_CLIP_UPPER}')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'NumberOfTime60-89DaysPastDueNotWorse', discrete=True,
                      title='', ax=axs[0])
plot_WoE(train_raw, 'NumberOfTime60-89DaysPastDueNotWorse', ax=axs[1])

fig.suptitle(fr'raw')
plt.tight_layout()

### Preprocessing

In [ ]:
train_processed['NumberOfTime60-89DaysPastDueNotWorse'] = \
    scale(train_raw['NumberOfTime60-89DaysPastDueNotWorse'].clip(upper=PD60_89_CLIP_UPPER))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'NumberOfTime60-89DaysPastDueNotWorse', max_value=2*PD60_89_CLIP_UPPER,
                      title=rf'raw $\leq {2*PD60_89_CLIP_UPPER}$', discrete=True, ax=axes[0])
plot_feature_analysis(train_processed, 'NumberOfTime60-89DaysPastDueNotWorse', discrete=True,
                      title=rf'preprocessed', ax=axes[1])

plt.tight_layout()

### WoE Transformation

In [ ]:
train_woe['NumberOfTime60-89DaysPastDueNotWorse'], pd60_89_binner = apply_woe_binning(
    df=train_raw,
    feature='NumberOfTime60-89DaysPastDueNotWorse',
    target=train_raw['SeriousDlqin2yrs'],
    max_n_bins=10,
    min_prebin_size=0.01,
    special_codes={98: 98, 96: 96},
    monotonic_trend='ascending'
)

In [ ]:
binning_table = pd60_89_binner.binning_table
binning_table.build()

In [ ]:
binning_table.plot(figsize=(6, 4))

## `NumberOfTimes90DaysLate`

In [ ]:
PD90_CLIP_UPPER_PCTL = 0.99
PD90_CLIP_UPPER = train_raw['NumberOfTimes90DaysLate'].quantile(PD90_CLIP_UPPER_PCTL)

print(f'{100*PD90_CLIP_UPPER_PCTL} процентиль: {PD90_CLIP_UPPER}')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'NumberOfTimes90DaysLate', discrete=True,
                      title='', ax=axs[0])
plot_WoE(train_raw, 'NumberOfTimes90DaysLate', ax=axs[1])

fig.suptitle(fr'raw')
plt.tight_layout()

### Preprocessing

In [ ]:
train_processed['NumberOfTimes90DaysLate'] = \
    scale(np.log1p(train_raw['NumberOfTimes90DaysLate'].clip(upper=PD90_CLIP_UPPER)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_feature_analysis(train_raw, 'NumberOfTimes90DaysLate', max_value=2*PD90_CLIP_UPPER,
                      title=rf'raw $\leq {2*PD90_CLIP_UPPER}$', discrete=True, ax=axes[0])
plot_feature_analysis(train_processed, 'NumberOfTimes90DaysLate', discrete=True,
                      title=rf'preprocessed', ax=axes[1])

plt.tight_layout()

### New Features

Агрегированные признаки на основе количества просрочек разной длительности:
- `has_delinquency` — бинарный индикатор наличия любой просрочки;
- `total_delinquencies` — общее число просрочек;
- `weighted_delinquencies` — взвешенная сумма, где просрочки 30–59, 60–89 и 90+ дней имеют веса 1, 2 и 3 соответственно.

In [ ]:
pd30 = train_raw["NumberOfTime30-59DaysPastDueNotWorse"].clip(upper=PD30_59_CLIP_UPPER)
pd60 = train_raw["NumberOfTime60-89DaysPastDueNotWorse"].clip(upper=PD60_89_CLIP_UPPER)
pd90 = train_raw["NumberOfTimes90DaysLate"].clip(upper=PD90_CLIP_UPPER)

train_processed["has_delinquency"] = ((pd30 + pd60 + pd90) > 0).astype(int)
train_processed["total_delinquencies"] = scale(pd30 + pd60 + pd90)
train_processed["weighted_delinquencies"] = scale(1*pd30 + 2*pd60 + 3*pd90)

### WoE Transformation

In [ ]:
train_woe['NumberOfTimes90DaysLate'], pd90_binner = apply_woe_binning(
    df=train_raw,
    feature='NumberOfTimes90DaysLate',
    target=train_raw['SeriousDlqin2yrs'],
    max_n_bins=10,
    min_prebin_size=0.005,
    special_codes={98: 98, 96: 96},
    monotonic_trend='ascending'
)

In [ ]:
binning_table = pd90_binner.binning_table
binning_table.build()

In [ ]:
binning_table.plot(figsize=(6, 4))

---

## Resulting DataFrames

In [ ]:
# Scale indicators to match the pipeline results
train_processed[['is_young', 'is_senior', 'NumberOfDependents_missing', 'MonthlyIncome_missing', 'MonthlyIncome_clipped', 'MonthlyIncome_zero', 'LowIncome', 'utilization_clipped', 'utilization_99999990', 'DebtRatio_low', 'DebtRatio_clipped', 'DebtRatio_zero', 'DebtRatio_whole', 'NumberOfDependents_missing', 'has_delinquency']] = scale(train_processed[['is_young', 'is_senior', 'NumberOfDependents_missing', 'MonthlyIncome_missing', 'MonthlyIncome_clipped', 'MonthlyIncome_zero', 'LowIncome', 'utilization_clipped', 'utilization_99999990', 'DebtRatio_low', 'DebtRatio_clipped', 'DebtRatio_zero', 'DebtRatio_whole', 'NumberOfDependents_missing', 'has_delinquency']])
train_processed.describe()

In [ ]:
train_woe.describe()